# 1D CNN — Two-Hand Dynamic Gesture Recognition (multi-variant)

Clean, single-path pipeline for training **several 1D-CNN architecture variants** on the glove data and exporting a comparison PDF.

**Pipeline stages**
1. Configuration (data paths, sensor selection, preprocessing, augmentation, training)
2. Sensor column selection
3. Load CSVs from `DynamicTrainingData/TwoSec`
4. Preprocess: resample to fixed length + Butterworth low-pass filter
5. Train/test split (hold-out)
6. Cross-validation with augmentation **inside** each fold (so the validation set is never augmented)
7. Final retrain on the full training pool and evaluation on hold-out
8. Batch test on `TestData/Jestin/TwoHandDynamic` using per-subfolder labels
9. Save model + log experiment row

Only the CNN-on-sequences path is supported here. Engineered-feature/MLP variants have been removed to keep the notebook readable.


## 1. Imports


In [1]:
import os
import re
import glob
import json
import hashlib
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal as scipy_signal
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', 20)
print('Imports OK.')


I0000 00:00:1778048210.081030   53576 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778048210.084574   53576 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778048210.338004   53576 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778048211.895071   53576 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENAB

Imports OK.


## 2. Configuration

All knobs in one place. Edit only this cell to run different experiments.


In [22]:
BATCH_TEST_ROOT = '/home/jestin/ThesisRepo/ML/NewTestData/Joselyn/Dynamic'

In [23]:
# ── Data locations ───────────────────────────────────────────────────────────
# You can include data from MULTIPLE users / source folders.
#
# Each entry in DATA_PATHS is a folder that DIRECTLY contains gesture-label
# subfolders (e.g. .../Alan/Dynamic, .../Bridgette/Dynamic). All matching label
# folders across these paths are merged by label name, so e.g.
# Alan/Dynamic/Wave and Bridgette/Dynamic/Wave end up in the same "Wave" class.
#
# Tip: comment out a path to exclude that user from training without deleting it.
DATA_PATHS = [
    '/home/jestin/ThesisRepo/ML/NewTestData/Alan/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/Harry/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/Alex/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/Bridgette/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/Georgia/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/Henry/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/Jestin/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/Joselyn/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/Josh/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/Marcus/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/Nat/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/Stephen/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/Tash/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/Val/Dynamic',
]

# Backwards-compat shortcut: set DATA_ROOT to a string and leave DATA_PATHS = None
# (or empty) to use the old single-folder workflow.
DATA_ROOT = None

INCLUDE_LABELS  = None   # None = use every label subfolder found

# ── Sensor selection ─────────────────────────────────────────────────────────
USE_LEFT_HAND  = True
USE_RIGHT_HAND = True

USE_YPR        = True   # yaw / pitch / roll (or heading/pitch/roll for wrist)
USE_QUAT       = False   # quaternion w/x/y/z
USE_ACCEL      = True   # ax / ay / az
USE_FLEX       = True   # mcp_flex / pip_flex (fingers only)

# Segments to include
USE_WRIST  = True
USE_PALM   = True
USE_THUMB  = True
USE_INDEX  = True
USE_MIDDLE = True
USE_RING   = True
USE_PINKY  = True

# ── Preprocessing ────────────────────────────────────────────────────────────
RESAMPLE_TO_N_STEPS    = 90
APPLY_BUTTERWORTH      = True
BUTTERWORTH_CUTOFF_HZ  = 10.0
BUTTERWORTH_ORDER      = 4
SAMPLING_RATE_HZ       = 30.0
NORMALISATION          = 'minmax'   # 'standard' | 'minmax' | None

# ── Train/test split ─────────────────────────────────────────────────────────
TEST_SIZE             = 0.10
RANDOM_STATE          = 42
STRATIFY_SPLIT        = True

# ── Cross-validation ─────────────────────────────────────────────────────────
RUN_CROSS_VALIDATION  = True
CV_FOLDS              = 5
CV_SHUFFLE            = True

# ── Augmentation (training only, inside each CV fold) ────────────────────────
AUGMENT_TRAINING_DATA          = True
AUGMENTATION_COPIES_PER_SAMPLE = 5
AUGMENTATION_RANDOM_SEED       = 42
AUGMENT_CONFIG = {
    'time_shift':      {'enabled': False,  'apply_prob': 0.7, 'max_shift_steps': 3, 'fill_mode': 'edge'},
    'time_warp':       {'enabled': False,  'apply_prob': 0.5, 'speed_range': (0.90, 1.10)},
    'time_mask':       {'enabled': False,  'apply_prob': 0.5, 'max_masks': 2, 'max_mask_size': 3, 'fill_mode': 'zero'},
    'gaussian_noise':  {'enabled': False,  'apply_prob': 1.0, 'std_ratio': 0.02},
    'amplitude_scale': {'enabled': True,  'apply_prob': 0.5, 'scale_range': (0.95, 1.05)},
    'baseline_offset': {'enabled': True,  'apply_prob': 0.5, 'offset_std_ratio': 0.02},
    'channel_dropout': {'enabled': False,  'apply_prob': 0.3, 'drop_fraction': 0.03, 'fill_mode': 'zero'},
    # Only enable left/right swap if the class label is symmetric under that swap
    'left_right_swap': {'enabled': False, 'apply_prob': 0.5},
}

# ── Training hyperparameters ─────────────────────────────────────────────────
TRAIN_EPOCHS           = 15
TRAIN_BATCH_SIZE       = 16
TRAIN_VALIDATION_SPLIT = 0.2   # only used by the final retrain (not by CV)
TRAIN_VERBOSE          = 1

# ── Output paths ─────────────────────────────────────────────────────────────
MODEL_OUTPUT_DIR     = Path('saved_models')
EXPERIMENT_LOG_PATH  = MODEL_OUTPUT_DIR / 'experiment_results.csv'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve active data paths (supports both DATA_PATHS list and legacy DATA_ROOT)
ACTIVE_DATA_PATHS = []
if DATA_PATHS:
    ACTIVE_DATA_PATHS = [p for p in DATA_PATHS if p and not p.strip().startswith('#')]
if not ACTIVE_DATA_PATHS and DATA_ROOT:
    ACTIVE_DATA_PATHS = [DATA_ROOT]

print('Configuration loaded.')
print(f'  Data paths ({len(ACTIVE_DATA_PATHS)}):')
for p in ACTIVE_DATA_PATHS:
    print(f'    - {p}')


Configuration loaded.
  Data paths (6):
    - /home/jestin/ThesisRepo/ML/NewTestData/Alan/Dynamic
    - /home/jestin/ThesisRepo/ML/NewTestData/Harry/Dynamic
    - /home/jestin/ThesisRepo/ML/NewTestData/Jestin/Dynamic
    - /home/jestin/ThesisRepo/ML/NewTestData/Joselyn/Dynamic
    - /home/jestin/ThesisRepo/ML/NewTestData/Marcus/Dynamic
    - /home/jestin/ThesisRepo/ML/NewTestData/Tash/Dynamic


## 3. Sensor column selection

Build the list of channel names that match the configuration toggles. The CSVs share a known column schema, so we generate names directly rather than parsing every header.


In [24]:
SEGMENTS_WITH_FLEX = ['thumb', 'index', 'middle', 'ring', 'pinky']  # palm has no flex


def build_sensor_columns(hands, segments, use_ypr, use_quat, use_accel, use_flex):
    cols = []
    for hand in hands:
        for seg in segments:
            if seg == 'wrist':
                # wrist has a single IMU; YPR fields are heading/pitch/roll
                p = f'{hand}_wrist'
                if use_ypr:   cols += [f'{p}_heading', f'{p}_pitch', f'{p}_roll']
                if use_quat:  cols += [f'{p}_quat_w', f'{p}_quat_x', f'{p}_quat_y', f'{p}_quat_z']
                if use_accel: cols += [f'{p}_ax', f'{p}_ay', f'{p}_az']
                # no flex on wrist
            else:
                for loc in ['mid', 'prox']:
                    p = f'{hand}_{seg}_{loc}'
                    if use_ypr:   cols += [f'{p}_yaw', f'{p}_pitch', f'{p}_roll']
                    if use_quat:  cols += [f'{p}_quat_w', f'{p}_quat_x', f'{p}_quat_y', f'{p}_quat_z']
                    if use_accel: cols += [f'{p}_ax', f'{p}_ay', f'{p}_az']
                if use_flex and seg in SEGMENTS_WITH_FLEX:
                    cols += [f'{hand}_{seg}_mcp_flex', f'{hand}_{seg}_pip_flex']
    return cols


resolved_hands = [h for h, on in [('left', USE_LEFT_HAND), ('right', USE_RIGHT_HAND)] if on]
resolved_segs = [s for s, on in [
    ('wrist', USE_WRIST), ('palm', USE_PALM), ('thumb', USE_THUMB),
    ('index', USE_INDEX), ('middle', USE_MIDDLE), ('ring', USE_RING), ('pinky', USE_PINKY)
] if on]

SENSOR_COLS = build_sensor_columns(
    hands=resolved_hands, segments=resolved_segs,
    use_ypr=USE_YPR, use_quat=USE_QUAT, use_accel=USE_ACCEL, use_flex=USE_FLEX,
)

print(f'Selected {len(SENSOR_COLS)} sensor columns.')
print('First 6:', SENSOR_COLS[:6])
print('Last  6:', SENSOR_COLS[-6:])


Selected 176 sensor columns.
First 6: ['left_wrist_heading', 'left_wrist_pitch', 'left_wrist_roll', 'left_wrist_ax', 'left_wrist_ay', 'left_wrist_az']
Last  6: ['right_pinky_prox_roll', 'right_pinky_prox_ax', 'right_pinky_prox_ay', 'right_pinky_prox_az', 'right_pinky_mcp_flex', 'right_pinky_pip_flex']


## 4. Load training data

Walk the label subfolders under `DATA_ROOT` and read each CSV as a `(T, C)` array of the selected sensor channels.


In [25]:
def load_dataset(data_paths, include_labels, sensor_cols):
    """
    Load all CSVs from gesture-label subfolders found inside each path in
    `data_paths`. Trials from the same label across different source folders
    are merged into a single class.

    Returns:
        trials      : list of np.ndarray (per-trial sensor matrices)
        labels      : list of class label strings
        class_names : sorted list of unique class labels
        source_info : list of source label-folder paths per trial (parallel to trials)
    """
    if isinstance(data_paths, str):
        data_paths = [data_paths]
    if not data_paths:
        raise ValueError('No data paths provided. Set DATA_PATHS or DATA_ROOT.')

    resolved_paths = []
    for p in data_paths:
        p_exp = os.path.expanduser(p)
        if not os.path.isdir(p_exp):
            raise FileNotFoundError(
                f"Data path not found: {p_exp!r}\n"
                "Update DATA_PATHS in the Configuration cell."
            )
        resolved_paths.append(p_exp)

    SKIP = {'PDF', 'sdb', 'idk'}
    label_to_folders = {}   # label -> list of full folder paths
    for root in resolved_paths:
        for d in sorted(os.listdir(root)):
            full = os.path.join(root, d)
            if not os.path.isdir(full) or d in SKIP:
                continue
            label_to_folders.setdefault(d, []).append(full)

    if include_labels is not None:
        label_to_folders = {k: v for k, v in label_to_folders.items()
                            if k in include_labels}
    if not label_to_folders:
        raise ValueError('No label folders found. Check DATA_PATHS and INCLUDE_LABELS.')

    label_names = sorted(label_to_folders.keys())
    print(f'Found {len(label_names)} gesture classes across {len(resolved_paths)} source folder(s):')

    trials, labels, source_info = [], [], []
    for label in label_names:
        folders = label_to_folders[label]
        per_source_counts = []
        for folder in folders:
            csv_files = sorted(glob.glob(os.path.join(folder, '*.csv')))
            per_source_counts.append((folder, len(csv_files)))
            for fpath in csv_files:
                try:
                    df = pd.read_csv(fpath)
                    available = [c for c in sensor_cols if c in df.columns]
                    if not available:
                        print(f'    WARNING: no matching sensor columns in {os.path.basename(fpath)}')
                        continue
                    trials.append(df[available].values.astype(np.float32))
                    labels.append(label)
                    source_info.append(folder)
                except Exception as e:
                    print(f'    ERROR loading {os.path.basename(fpath)}: {e}')
        total = sum(c for _, c in per_source_counts)
        extra = ''
        if len(per_source_counts) > 1:
            extra = ' from ' + ', '.join(
                f"{os.path.basename(os.path.dirname(fp))}({n})"
                for fp, n in per_source_counts
            )
        print(f'  [{label}]  →  {total} files{extra}')

    print(f'\nTotal trials loaded: {len(trials)}')
    return trials, labels, label_names, source_info


trials_raw, labels_raw, class_names, source_info = load_dataset(
    ACTIVE_DATA_PATHS, INCLUDE_LABELS, SENSOR_COLS
)

print('\nClass distribution:')
for cls, cnt in sorted(Counter(labels_raw).items()):
    print(f'  {cls}: {cnt} trials')

# Per-source-folder distribution
print('\nTrials per source folder:')
for src, cnt in sorted(Counter(source_info).items()):
    tag = os.sep.join(src.split(os.sep)[-3:])
    print(f'  {tag}: {cnt} trials')


Found 7 gesture classes across 6 source folder(s):
  [Double_Lower]  →  25 files from Dynamic(5), Dynamic(5), Dynamic(5), Dynamic(0), Dynamic(5), Dynamic(5)
  [Double_Nothing]  →  20 files from Dynamic(5), Dynamic(5), Dynamic(0), Dynamic(0), Dynamic(5), Dynamic(5)
  [Double_Pistol_Recoil]  →  25 files from Dynamic(5), Dynamic(5), Dynamic(5), Dynamic(0), Dynamic(5), Dynamic(5)
  [Double_Raise]  →  25 files from Dynamic(5), Dynamic(5), Dynamic(5), Dynamic(0), Dynamic(5), Dynamic(5)
  [Double_Wiggle]  →  24 files from Dynamic(5), Dynamic(5), Dynamic(4), Dynamic(0), Dynamic(5), Dynamic(5)
  [Double_cmere]  →  30 files from Dynamic(5), Dynamic(5), Dynamic(5), Dynamic(5), Dynamic(5), Dynamic(5)
  [Drum_Roll]  →  20 files from Dynamic(5), Dynamic(5), Dynamic(0), Dynamic(0), Dynamic(5), Dynamic(5)

Total trials loaded: 169

Class distribution:
  Double_Lower: 25 trials
  Double_Nothing: 20 trials
  Double_Pistol_Recoil: 25 trials
  Double_Raise: 25 trials
  Double_Wiggle: 24 trials
  Double_cm

## 5. Preprocessing

1. Resample every trial to a fixed length so the CNN gets a constant input shape.
2. Optional zero-phase Butterworth low-pass filter to smooth high-frequency noise.


In [26]:
def resample_trial(trial, n_steps):
    """Linear resample (T, C) → (n_steps, C)."""
    T, C = trial.shape
    if T == n_steps:
        return trial.astype(np.float32, copy=False)
    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, n_steps)
    out = np.zeros((n_steps, C), dtype=np.float32)
    for c in range(C):
        out[:, c] = np.interp(new_idx, old_idx, trial[:, c])
    return out


def apply_butterworth(trials, cutoff, order, fs):
    """Zero-phase low-pass filter applied per channel."""
    nyq = fs / 2.0
    norm_cutoff = cutoff / nyq
    if norm_cutoff >= 1.0:
        print(f'  WARNING: cutoff {cutoff} Hz >= Nyquist {nyq} Hz — skipping filter.')
        return trials
    b, a = scipy_signal.butter(order, norm_cutoff, btype='low', analog=False)
    return [scipy_signal.filtfilt(b, a, t, axis=0).astype(np.float32) for t in trials]


# Resample
if RESAMPLE_TO_N_STEPS is not None:
    trials_resampled = [resample_trial(t, RESAMPLE_TO_N_STEPS) for t in trials_raw]
    print(f'Resampled all trials to {RESAMPLE_TO_N_STEPS} time steps.')
else:
    trials_resampled = trials_raw

# Filter
if APPLY_BUTTERWORTH:
    trials_filtered = apply_butterworth(trials_resampled, BUTTERWORTH_CUTOFF_HZ,
                                        BUTTERWORTH_ORDER, SAMPLING_RATE_HZ)
    print(f'Butterworth low-pass: cutoff={BUTTERWORTH_CUTOFF_HZ} Hz, order={BUTTERWORTH_ORDER}.')
else:
    trials_filtered = trials_resampled
    print('Butterworth filter skipped.')

sequence_length, n_channels = trials_filtered[0].shape
print(f'Trial shape: ({sequence_length}, {n_channels})  (time_steps × channels)')


Resampled all trials to 90 time steps.
Butterworth low-pass: cutoff=10.0 Hz, order=4.
Trial shape: (90, 176)  (time_steps × channels)


## 6. Build X, y and the train/test split

We stack all trials into a single `(N, T, C)` tensor, encode labels, then carve off a stratified hold-out test set. The hold-out set is **never** seen during training, augmentation, or scaler fitting.


In [27]:
X = np.stack(trials_filtered, axis=0).astype(np.float32)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

le = LabelEncoder()
y = le.fit_transform(labels_raw)
n_classes = len(le.classes_)
print(f'X shape: {X.shape}    y shape: {y.shape}    n_classes: {n_classes}')
print('Label mapping:', dict(zip(le.classes_, range(n_classes))))

X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y if STRATIFY_SPLIT and n_classes > 1 else None,
)
print(f'Train pool: {X_train_pool.shape}    Hold-out test: {X_test.shape}')


X shape: (169, 90, 176)    y shape: (169,)    n_classes: 7
Label mapping: {np.str_('Double_Lower'): 0, np.str_('Double_Nothing'): 1, np.str_('Double_Pistol_Recoil'): 2, np.str_('Double_Raise'): 3, np.str_('Double_Wiggle'): 4, np.str_('Double_cmere'): 5, np.str_('Drum_Roll'): 6}
Train pool: (152, 90, 176)    Hold-out test: (17, 90, 176)


## 7. Augmentation helpers

Each augmentation operates on a single `(T, C)` trial and is applied probabilistically per `AUGMENT_CONFIG`. `augment_training_set` produces `copies_per_sample` augmented variants per original trial and concatenates them with the originals.

Augmentation is applied **inside** each CV fold's training split — the fold's validation split is left untouched, just like the hold-out test set.


In [28]:
def _resize_linear(trial, target_steps):
    T, C = trial.shape
    if T == target_steps:
        return trial.astype(np.float32, copy=False)
    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, target_steps)
    out = np.zeros((target_steps, C), dtype=np.float32)
    for c in range(C):
        out[:, c] = np.interp(new_idx, old_idx, trial[:, c])
    return out


def aug_time_shift(trial, max_shift_steps=3, fill_mode='edge', rng=None):
    rng = rng or np.random.default_rng()
    if max_shift_steps <= 0:
        return trial.copy()
    shift = int(rng.integers(-max_shift_steps, max_shift_steps + 1))
    if shift == 0:
        return trial.copy()
    out = np.empty_like(trial)
    out[:] = 0.0 if fill_mode == 'zero' else (trial[0] if shift > 0 else trial[-1])
    if shift > 0:
        out[shift:] = trial[:-shift]
    else:
        out[:shift] = trial[-shift:]
    return out


def aug_time_warp(trial, speed_range=(0.9, 1.1), rng=None):
    rng = rng or np.random.default_rng()
    factor = float(rng.uniform(*speed_range))
    T = trial.shape[0]
    warped_steps = max(4, int(round(T * factor)))
    return _resize_linear(_resize_linear(trial, warped_steps), T)


def aug_time_mask(trial, max_masks=2, max_mask_size=3, fill_mode='zero', rng=None):
    rng = rng or np.random.default_rng()
    out = trial.copy()
    T, C = out.shape
    n_masks = int(rng.integers(1, max_masks + 1)) if max_masks > 0 else 0
    for _ in range(n_masks):
        size = int(rng.integers(1, max_mask_size + 1))
        start = int(rng.integers(0, max(1, T - size + 1)))
        if fill_mode == 'mean':
            out[start:start+size] = out.mean(axis=0, keepdims=True)
        elif fill_mode == 'noise':
            ch_std = np.std(out, axis=0, keepdims=True)
            out[start:start+size] = rng.normal(0.0, np.maximum(ch_std, 1e-6), size=(size, C))
        else:
            out[start:start+size] = 0.0
    return out


def aug_gaussian_noise(trial, std_ratio=0.02, rng=None):
    rng = rng or np.random.default_rng()
    ch_std = np.std(trial, axis=0, keepdims=True)
    noise_std = np.maximum(ch_std * std_ratio, 1e-6)
    return (trial + rng.normal(0.0, noise_std, size=trial.shape)).astype(np.float32)


def aug_amplitude_scale(trial, scale_range=(0.95, 1.05), rng=None):
    rng = rng or np.random.default_rng()
    scale = rng.uniform(scale_range[0], scale_range[1], size=(1, trial.shape[1]))
    return (trial * scale).astype(np.float32)


def aug_baseline_offset(trial, offset_std_ratio=0.02, rng=None):
    rng = rng or np.random.default_rng()
    ch_std = np.std(trial, axis=0, keepdims=True)
    offset_std = np.maximum(ch_std * offset_std_ratio, 1e-6)
    offset = rng.normal(0.0, offset_std, size=(1, trial.shape[1]))
    return (trial + offset).astype(np.float32)


def aug_channel_dropout(trial, drop_fraction=0.03, fill_mode='zero', rng=None):
    rng = rng or np.random.default_rng()
    out = trial.copy()
    C = out.shape[1]
    n_drop = max(1, int(round(C * drop_fraction))) if drop_fraction > 0 else 0
    if n_drop == 0:
        return out
    idx = rng.choice(C, size=min(n_drop, C), replace=False)
    if fill_mode == 'mean':
        out[:, idx] = out.mean(axis=0, keepdims=True)[:, idx]
    else:
        out[:, idx] = 0.0
    return out


def aug_left_right_swap(trial, sensor_cols, rng=None):
    out = trial.copy()
    name_to_idx = {col: i for i, col in enumerate(sensor_cols)}
    for col in sensor_cols:
        if col.startswith('left'):
            partner = 'right' + col[4:]
            if partner in name_to_idx:
                i, j = name_to_idx[col], name_to_idx[partner]
                out[:, i], out[:, j] = trial[:, j], trial[:, i]
    return out


def apply_augmentations(trial, sensor_cols, config, rng):
    """Apply each augmentation independently with its own probability."""
    out = trial.astype(np.float32, copy=True)

    def _drop_meta(cfg):
        return {k: v for k, v in cfg.items() if k not in ('enabled', 'apply_prob')}

    def _maybe(name, fn, **extra):
        cfg = config.get(name, {})
        if cfg.get('enabled', False) and rng.random() < cfg.get('apply_prob', 1.0):
            return fn(out, **_drop_meta(cfg), **extra, rng=rng)
        return out

    out = _maybe('time_shift',      aug_time_shift)
    out = _maybe('time_warp',       aug_time_warp)
    out = _maybe('time_mask',       aug_time_mask)
    out = _maybe('gaussian_noise',  aug_gaussian_noise)
    out = _maybe('amplitude_scale', aug_amplitude_scale)
    out = _maybe('baseline_offset', aug_baseline_offset)
    out = _maybe('channel_dropout', aug_channel_dropout)
    if config.get('left_right_swap', {}).get('enabled', False) \
            and rng.random() < config['left_right_swap'].get('apply_prob', 1.0):
        out = aug_left_right_swap(out, sensor_cols=sensor_cols, rng=rng)

    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def augment_training_set(X_tr, y_tr, sensor_cols, config, copies_per_sample=1, random_state=42):
    if copies_per_sample <= 0 or len(X_tr) == 0:
        return X_tr, y_tr
    rng = np.random.default_rng(random_state)
    X_aug, y_aug = [], []
    for trial, label in zip(X_tr, y_tr):
        for _ in range(copies_per_sample):
            X_aug.append(apply_augmentations(trial, sensor_cols, config, rng))
            y_aug.append(label)
    X_combined = np.concatenate([X_tr, np.stack(X_aug, axis=0)], axis=0)
    y_combined = np.concatenate([y_tr, np.array(y_aug, dtype=y_tr.dtype)], axis=0)
    return X_combined, y_combined


## 8. CNN model and scaler helper

`scale_train_val_test` fits the scaler on the training fold only, then applies it to validation and to a copy of the hold-out test set so we always have a correctly-scaled view of the test set per fold.


In [29]:
def build_cnn_model(sequence_length, n_channels, n_classes):
    model = Sequential([
        Conv1D(32, kernel_size=4, activation='relu', input_shape=(sequence_length, n_channels)),
        MaxPooling1D(pool_size=2),
        Conv1D(64, kernel_size=4, activation='relu'),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


def make_scaler(kind):
    if kind == 'standard':
        return StandardScaler()
    if kind == 'minmax':
        return MinMaxScaler()
    return None


def scale_sets(X_train, X_val, X_test, kind):
    """Fit on train only; transform val and test."""
    scaler = make_scaler(kind)
    if scaler is None:
        return X_train, X_val, X_test, None
    Nt, T, C = X_train.shape
    Xt = scaler.fit_transform(X_train.reshape(Nt, T * C)).reshape(Nt, T, C).astype(np.float32)
    Xv = scaler.transform(X_val.reshape(X_val.shape[0], T * C)).reshape(X_val.shape[0], T, C).astype(np.float32) \
         if len(X_val) else X_val
    Xs = scaler.transform(X_test.reshape(X_test.shape[0], T * C)).reshape(X_test.shape[0], T, C).astype(np.float32) \
         if len(X_test) else X_test
    return Xt, Xv, Xs, scaler


## 8b. CNN architecture variants

We compare several 1D-CNN architectures using the same data, splits, CV folds, augmentation, scaler, and training loop. Each entry in `VARIANTS` is `(name, description, builder_fn)`. Add or remove entries here to change what gets compared — every downstream cell loops over this list.

> Variant names must be valid Keras model scope names (letters, digits, `_`, `.`, `/`, `-`, `>`). Avoid `+`, spaces, etc.


In [ ]:
from tensorflow.keras.layers import (
    BatchNormalization, GlobalAveragePooling1D, Activation, Input,
)
from tensorflow.keras.models import Sequential


def build_baseline(seq_len, n_chan, n_classes):
    """Original architecture from the notebook."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 4, activation='relu'),
        MaxPooling1D(2),
        Conv1D(64, 4, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='Baseline')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_shallow(seq_len, n_chan, n_classes):
    """Single Conv block — fewer params, faster to train."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 5, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='Shallow')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_deep(seq_len, n_chan, n_classes):
    """Three Conv blocks — more representational depth."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 4, activation='relu'),
        MaxPooling1D(2),
        Conv1D(64, 4, activation='relu'),
        MaxPooling1D(2),
        Conv1D(128, 3, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.4),
        Dense(n_classes, activation='softmax'),
    ], name='Deep')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_bn_gap(seq_len, n_chan, n_classes):
    """BatchNorm + GlobalAveragePooling — typically better generalisation, fewer params."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 4, padding='same'),
        BatchNormalization(), Activation('relu'),
        MaxPooling1D(2),
        Conv1D(64, 4, padding='same'),
        BatchNormalization(), Activation('relu'),
        MaxPooling1D(2),
        Conv1D(128, 3, padding='same'),
        BatchNormalization(), Activation('relu'),
        GlobalAveragePooling1D(),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='BN_GAP')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_wide_kernel(seq_len, n_chan, n_classes):
    """Wider kernels — larger temporal receptive field per layer."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 8, activation='relu'),
        MaxPooling1D(2),
        Conv1D(64, 8, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='WideKernel')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


VARIANTS = [
    ('Baseline',   'Conv1D(32,k=4)→Pool→Conv1D(64,k=4)→Pool→Flatten→Dense(64)→Dropout(0.3)→Softmax', build_baseline),
    ('Shallow',    'Single Conv block: Conv1D(32,k=5)→Pool→Flatten→Dense(32)→Dropout(0.3)→Softmax',           build_shallow),
    ('Deep',       'Three Conv blocks: Conv1D(32,4)→Conv1D(64,4)→Conv1D(128,3)→Flatten→Dense(128)→Dropout(0.4)→Softmax', build_deep),
    ('BN_GAP',     'Conv+BN+ReLU ×3 with GlobalAveragePooling1D head — fewer params, less overfit',                build_bn_gap),
    ('WideKernel', 'Conv1D(32,k=8)→Pool→Conv1D(64,k=8)→Pool→Flatten→Dense(64)→Dropout(0.3)→Softmax', build_wide_kernel),
]

# Container populated by the CV / retrain / hold-out / batch-test cells below.
# Each entry: dict with name, description, n_params, cv_results, cv_mean, cv_std,
# history, test_accuracy, test_loss, classification_report, y_pred, batch_df,
# batch_correct, batch_total, batch_accuracy, model, model_path, scaler.
variant_results = []
print(f'Configured {len(VARIANTS)} variants:')
for n, d, _ in VARIANTS:
    print(f'  - {n}: {d}')


## 9. Cross-validation (augmentation inside each fold)

For every fold:
1. Split `X_train_pool` into fold train / fold val.
2. Augment **only** fold train.
3. Fit scaler on fold train; transform fold val and the (separate) hold-out test.
4. Train a fresh CNN; record validation accuracy.

We keep the best-fold model as a starting point, but the final reported test accuracy comes from a clean retrain on the full training pool (next cell).


In [ ]:
# Per-variant cross-validation. Reuses augment_training_set / scale_sets / build_*
# from the helper cells above. The familiar legacy variables (cv_histories,
# cv_fold_results, cv_mean_accuracy, cv_std_accuracy, best_cv_fold_index) end up
# pointing at the LAST variant trained, so existing downstream cells still run.
import tensorflow as tf

variant_results = []  # reset on re-run

for v_name, v_desc, v_builder in VARIANTS:
    print(f'\n{"="*72}\n  Variant: {v_name}\n{"="*72}')
    cv_histories = []
    cv_fold_results = []
    cv_mean_accuracy = None
    cv_std_accuracy = None
    best_cv_fold_index = None

    if RUN_CROSS_VALIDATION and CV_FOLDS > 1:
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=CV_SHUFFLE,
                              random_state=RANDOM_STATE if CV_SHUFFLE else None)
        for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_train_pool, y_train_pool), start=1):
            X_tr, y_tr = X_train_pool[tr_idx].copy(), y_train_pool[tr_idx].copy()
            X_va, y_va = X_train_pool[va_idx].copy(), y_train_pool[va_idx].copy()

            if AUGMENT_TRAINING_DATA:
                X_tr, y_tr = augment_training_set(
                    X_tr, y_tr,
                    sensor_cols=SENSOR_COLS,
                    config=AUGMENT_CONFIG,
                    copies_per_sample=AUGMENTATION_COPIES_PER_SAMPLE,
                    random_state=AUGMENTATION_RANDOM_SEED + fold_idx,
                )

            X_tr_s, X_va_s, _, _ = scale_sets(X_tr, X_va, X_test, NORMALISATION)

            tf.keras.backend.clear_session()
            tf.random.set_seed(RANDOM_STATE + fold_idx)
            fold_model = v_builder(sequence_length, n_channels, n_classes)
            history = fold_model.fit(
                X_tr_s, y_tr,
                epochs=TRAIN_EPOCHS, batch_size=TRAIN_BATCH_SIZE,
                validation_data=(X_va_s, y_va), verbose=TRAIN_VERBOSE,
            )
            val_pred = np.argmax(fold_model.predict(X_va_s, verbose=0), axis=1)
            val_acc = float(accuracy_score(y_va, val_pred))

            cv_histories.append(history.history)
            cv_fold_results.append({
                'fold': fold_idx,
                'train_after_aug': int(X_tr.shape[0]),
                'val_samples': int(len(va_idx)),
                'val_accuracy': val_acc,
                'best_val_accuracy': float(max(history.history.get('val_accuracy', [val_acc]))),
                'final_val_accuracy': float(history.history.get('val_accuracy', [val_acc])[-1]),
            })
            print(f'  Fold {fold_idx}/{CV_FOLDS}: val_acc={val_acc:.4f}')

        val_accs = [r['val_accuracy'] for r in cv_fold_results]
        cv_mean_accuracy = float(np.mean(val_accs))
        cv_std_accuracy  = float(np.std(val_accs))
        best_cv_fold_index = int(np.argmax(val_accs)) + 1
        print(f'  CV mean: {cv_mean_accuracy:.4f} ± {cv_std_accuracy:.4f}, best fold {best_cv_fold_index}')

    variant_results.append({
        'name': v_name,
        'description': v_desc,
        'builder': v_builder,
        'cv_histories': cv_histories,
        'cv_fold_results': cv_fold_results,
        'cv_mean': cv_mean_accuracy,
        'cv_std':  cv_std_accuracy,
        'best_cv_fold_index': best_cv_fold_index,
    })


## 10. Final retrain on the full training pool

CV gave us an honest estimate of generalisation. Now we retrain on the whole training pool with augmentation, fit a fresh scaler, and evaluate on the hold-out test set. This `model` and `scaler` are what get used for batch testing.


In [ ]:
# Final retrain on the full training pool — once per variant. The familiar
# `model`, `scaler`, `history`, `X_train_final`, `y_train_final`, `X_test_final`
# variables end up pointing at the LAST variant for compatibility with cells
# 23/25/27 if you ever decide to skip the variants loop and run a single one.
import tensorflow as tf

# Build the training set once (same data + augmentation seed for fair comparison)
if AUGMENT_TRAINING_DATA:
    X_tr_full, y_tr_full = augment_training_set(
        X_train_pool, y_train_pool,
        sensor_cols=SENSOR_COLS,
        config=AUGMENT_CONFIG,
        copies_per_sample=AUGMENTATION_COPIES_PER_SAMPLE,
        random_state=AUGMENTATION_RANDOM_SEED,
    )
else:
    X_tr_full, y_tr_full = X_train_pool.copy(), y_train_pool.copy()
print(f'Final training set size: {X_tr_full.shape[0]} (orig pool {X_train_pool.shape[0]})')

for entry in variant_results:
    print(f'\n── Final retrain: {entry["name"]} ──')

    # Fresh scaler per variant (fit on full augmented pool, applied to hold-out)
    X_tr_s, _, X_test_s, scaler_v = scale_sets(X_tr_full, X_tr_full[:0], X_test, NORMALISATION)

    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_STATE)
    model_v = entry['builder'](sequence_length, n_channels, n_classes)
    n_params = int(model_v.count_params())
    history_v = model_v.fit(
        X_tr_s, y_tr_full,
        epochs=TRAIN_EPOCHS, batch_size=TRAIN_BATCH_SIZE,
        validation_split=TRAIN_VALIDATION_SPLIT, verbose=TRAIN_VERBOSE,
    )

    entry.update({
        'model':   model_v,
        'scaler':  scaler_v,
        'history': history_v.history,
        'n_params': n_params,
        'X_test_final': X_test_s,
    })
    print(f'  trainable params: {n_params:,}')

# Legacy aliases pointing at the last variant
_last = variant_results[-1]
model         = _last['model']
scaler        = _last['scaler']
history       = type('H', (), {'history': _last['history']})()
X_train_final = X_tr_s
y_train_final = y_tr_full
X_test_final  = _last['X_test_final']


## 11. Training curves


In [ ]:
# Overlay validation curves for all variants
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
for entry in variant_results:
    h = entry.get('history', {})
    if 'val_accuracy' in h:
        plt.plot(h['val_accuracy'], label=entry['name'])
plt.title('Validation accuracy (final retrain)')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(fontsize=8); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
for entry in variant_results:
    h = entry.get('history', {})
    if 'val_loss' in h:
        plt.plot(h['val_loss'], label=entry['name'])
plt.title('Validation loss (final retrain)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(fontsize=8); plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_OUTPUT_DIR / 'training_curves.png', dpi=140, bbox_inches='tight')
plt.show()


## 12. Hold-out test evaluation


In [ ]:
# Hold-out evaluation per variant
for entry in variant_results:
    test_loss_v, test_accuracy_v = entry['model'].evaluate(
        entry['X_test_final'], y_test, verbose=0
    )
    y_pred_v = np.argmax(entry['model'].predict(entry['X_test_final'], verbose=0), axis=1)
    rep_v = classification_report(
        y_test, y_pred_v, target_names=le.classes_, output_dict=True, zero_division=0
    )
    entry.update({
        'test_loss': float(test_loss_v),
        'test_accuracy': float(test_accuracy_v),
        'y_pred': y_pred_v,
        'classification_report': rep_v,
    })
    print(f'{entry["name"]:<12} test_loss={test_loss_v:.4f}  test_acc={test_accuracy_v:.4f}')

print('\nConfusion matrix (last variant):')
print(confusion_matrix(y_test, variant_results[-1]['y_pred']))
print('\nClassification report (last variant):')
print(classification_report(
    y_test, variant_results[-1]['y_pred'], target_names=le.classes_, zero_division=0
))

# Legacy aliases for downstream cells
test_loss     = variant_results[-1]['test_loss']
test_accuracy = variant_results[-1]['test_accuracy']
y_pred        = variant_results[-1]['y_pred']


## 13. Batch test on TestData/Jestin/TwoHandDynamic

The test folder contains one subfolder per gesture (matching the training class names, sometimes with case differences). We use **the subfolder name** as the ground-truth label — this is far more robust than parsing filenames. Class-name matching is case-insensitive so e.g. `TwoHandDynamic_L_Fist_R_LightBulb` (test) maps to `TwoHandDynamic_L_Fist_R_Lightbulb` (training).


In [34]:
def preprocess_csv_for_cnn(fpath, sensor_cols, resample_n, apply_butter, cutoff, order, fs, scaler,
                            seq_len, n_chan):
    """Read one CSV → (1, seq_len, n_chan) tensor ready for model.predict."""
    df = pd.read_csv(fpath)
    available = [c for c in sensor_cols if c in df.columns]
    if not available:
        raise ValueError('No matching sensor columns')
    if len(available) != len(sensor_cols):
        # Pad missing columns with zeros so the channel order matches training
        data = np.zeros((len(df), len(sensor_cols)), dtype=np.float32)
        idx_map = {c: i for i, c in enumerate(sensor_cols)}
        for c in available:
            data[:, idx_map[c]] = df[c].values.astype(np.float32)
    else:
        data = df[sensor_cols].values.astype(np.float32)

    if resample_n is not None:
        data = resample_trial(data, resample_n)
    if apply_butter:
        data = apply_butterworth([data], cutoff, order, fs)[0]
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

    if scaler is not None:
        data = scaler.transform(data.reshape(1, -1)).reshape(seq_len, n_chan).astype(np.float32)

    return data.reshape(1, seq_len, n_chan)


def build_test_label_map(class_names):
    """Case-insensitive map from test-folder class name → training class name."""
    m = {}
    for cls in class_names:
        m[cls.lower()] = cls
    return m


def run_batch_test(test_root, model, scaler, le, sensor_cols, class_names,
                   resample_n, apply_butter, cutoff, order, fs, seq_len, n_chan):
    test_root = os.path.expanduser(test_root)
    if not os.path.isdir(test_root):
        raise FileNotFoundError(f'BATCH_TEST_ROOT not found: {test_root!r}')

    label_map = build_test_label_map(class_names)
    sub_dirs = sorted(d for d in os.listdir(test_root)
                      if os.path.isdir(os.path.join(test_root, d)))
    if not sub_dirs:
        print(f'No subfolders found in {test_root}')
        return None

    correct, total, unrecognised = 0, 0, 0
    per_class = {}
    rows = []
    print(f'Batch testing files from {len(sub_dirs)} subfolders under {test_root}')
    print('-' * 100)

    for sub in sub_dirs:
        true_label = label_map.get(sub.lower())
        sub_path = os.path.join(test_root, sub)
        csv_files = sorted(f for f in glob.glob(os.path.join(sub_path, '*.csv'))
                           if not os.path.basename(f).startswith('_'))
        if true_label is None:
            print(f'  ? subfolder {sub!r} has no matching training class — skipping ({len(csv_files)} files)')
            unrecognised += len(csv_files)
            continue

        per_class.setdefault(true_label, {'correct': 0, 'total': 0})

        for fpath in csv_files:
            fname = os.path.basename(fpath)
            try:
                x = preprocess_csv_for_cnn(
                    fpath, sensor_cols,
                    resample_n, apply_butter, cutoff, order, fs,
                    scaler, seq_len, n_chan,
                )
                probs = model.predict(x, verbose=0)[0]
                pred_idx = int(np.argmax(probs))
                pred = le.inverse_transform([pred_idx])[0]
                conf = float(probs.max())
                ok = (pred == true_label)
                correct += int(ok); total += 1
                per_class[true_label]['correct'] += int(ok)
                per_class[true_label]['total']   += 1
                rows.append({'file': fname, 'true': true_label, 'pred': pred,
                             'confidence': conf, 'correct': ok})
                tag = '✓' if ok else '✗'
                t_short = true_label.replace('TwoHandDynamic_', '')
                p_short = pred.replace('TwoHandDynamic_', '')
                print(f'  {tag} {fname[:50]:<52}  true={t_short:<28}  pred={p_short:<28}  conf={conf:.3f}')
            except Exception as e:
                print(f'  ERROR {fname}: {e}')

    print('-' * 100)
    if total > 0:
        print(f'Overall batch accuracy: {correct/total:.4f}  ({correct}/{total})')
    if unrecognised:
        print(f'Files in unmatched subfolders: {unrecognised}')

    print('\nPer-class accuracy:')
    for cls, d in sorted(per_class.items()):
        if d['total']:
            print(f'  {cls:<55} {d["correct"]/d["total"]:.4f}  ({d["correct"]}/{d["total"]})')

    return pd.DataFrame(rows), correct, total




In [ ]:
# Batch test per variant (reuses run_batch_test defined above)
for entry in variant_results:
    print(f'\n── Batch test: {entry["name"]} ──')
    res = run_batch_test(
        BATCH_TEST_ROOT, entry['model'], entry['scaler'], le,
        SENSOR_COLS, class_names,
        RESAMPLE_TO_N_STEPS, APPLY_BUTTERWORTH, BUTTERWORTH_CUTOFF_HZ, BUTTERWORTH_ORDER,
        SAMPLING_RATE_HZ, sequence_length, n_channels,
    )
    if res is not None:
        df_v, c_v, t_v = res
        entry['batch_df'] = df_v
        entry['batch_correct'] = int(c_v)
        entry['batch_total'] = int(t_v)
        entry['batch_accuracy'] = float(c_v / t_v) if t_v else None
    else:
        entry['batch_df'] = None
        entry['batch_correct'] = 0
        entry['batch_total'] = 0
        entry['batch_accuracy'] = None

print('\n=== Summary across variants ===')
print(f'{"Variant":<12}  {"params":>10}  {"CV":>16}  {"hold-out":>10}  {"batch":>10}')
for e in variant_results:
    cv = f'{e["cv_mean"]:.3f}±{e["cv_std"]:.3f}' if e["cv_mean"] is not None else '—'
    bt = f'{e["batch_accuracy"]:.3f}' if e["batch_accuracy"] is not None else '—'
    print(f'{e["name"]:<12}  {e["n_params"]:>10,}  {cv:>16}  {e["test_accuracy"]:>10.3f}  {bt:>10}')

# Legacy aliases
batch_df = variant_results[-1].get('batch_df')
batch_correct = variant_results[-1].get('batch_correct', 0)
batch_total = variant_results[-1].get('batch_total', 0)
batch_test_accuracy = variant_results[-1].get('batch_accuracy')


## 14. Save model and append experiment row to CSV log


In [36]:
def _to_jsonable(obj):
    if isinstance(obj, dict):  return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):    return obj.tolist()
    if isinstance(obj, np.integer):    return int(obj)
    if isinstance(obj, np.floating):   return float(obj)
    if isinstance(obj, np.bool_):      return bool(obj)
    return obj


def _stable_json(obj):
    return json.dumps(_to_jsonable(obj), sort_keys=True, separators=(',', ':'))


def build_experiment_row():
    hist = {k: [float(v) for v in vs] for k, vs in history.history.items()}
    row = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'data_paths': _stable_json([str(p) for p in (ACTIVE_DATA_PATHS or [])]),
        'data_paths_count': len(ACTIVE_DATA_PATHS or []),
        'batch_test_root': str(BATCH_TEST_ROOT),
        'sensor_cols_count': len(SENSOR_COLS),
        'sensor_cols': _stable_json(SENSOR_COLS),
        'resample_to_n_steps': RESAMPLE_TO_N_STEPS,
        'apply_butterworth': bool(APPLY_BUTTERWORTH),
        'butterworth_cutoff_hz': BUTTERWORTH_CUTOFF_HZ,
        'butterworth_order': BUTTERWORTH_ORDER,
        'sampling_rate_hz': SAMPLING_RATE_HZ,
        'normalisation': NORMALISATION,
        'test_size': float(TEST_SIZE),
        'random_state': int(RANDOM_STATE),
        'cv_folds': int(CV_FOLDS),
        'run_cross_validation': bool(RUN_CROSS_VALIDATION),
        'cv_shuffle': bool(CV_SHUFFLE),
        'augment_training_data': bool(AUGMENT_TRAINING_DATA),
        'augmentation_copies_per_sample': int(AUGMENTATION_COPIES_PER_SAMPLE),
        'augmentation_random_seed': int(AUGMENTATION_RANDOM_SEED),
        'augmentation_config': _stable_json(AUGMENT_CONFIG),
        'n_classes': int(n_classes),
        'class_names': _stable_json(list(le.classes_)),
        'n_total_samples': int(X.shape[0]),
        'n_train_pool_samples': int(X_train_pool.shape[0]),
        'n_train_after_aug': int(X_train_final.shape[0]),
        'n_test_samples': int(X_test.shape[0]),
        'sequence_length': int(sequence_length),
        'n_channels': int(n_channels),
        'epochs': int(TRAIN_EPOCHS),
        'batch_size': int(TRAIN_BATCH_SIZE),
        'validation_split': float(TRAIN_VALIDATION_SPLIT),
        'cv_mean_accuracy': cv_mean_accuracy,
        'cv_std_accuracy': cv_std_accuracy,
        'cv_best_fold_index': best_cv_fold_index,
        'cv_fold_results_json': _stable_json(cv_fold_results),
        'cv_histories_json': _stable_json(cv_histories),
        'final_train_accuracy': float(history.history['accuracy'][-1]),
        'final_val_accuracy':   float(history.history['val_accuracy'][-1]),
        'final_train_loss':     float(history.history['loss'][-1]),
        'final_val_loss':       float(history.history['val_loss'][-1]),
        'best_val_accuracy':    float(max(history.history['val_accuracy'])),
        'best_val_loss':        float(min(history.history['val_loss'])),
        'test_accuracy':        float(test_accuracy),
        'test_loss':            float(test_loss),
        'batch_test_accuracy':  float(batch_test_accuracy) if batch_test_accuracy is not None else None,
        'history_json': _stable_json(hist),
    }
    sig_payload = {k: v for k, v in row.items() if k != 'timestamp'}
    row['row_signature'] = hashlib.sha256(_stable_json(sig_payload).encode()).hexdigest()
    return row


def append_experiment_row(csv_path=EXPERIMENT_LOG_PATH):
    row = build_experiment_row()
    df_new = pd.DataFrame([row])
    if csv_path.exists():
        df_existing = pd.read_csv(csv_path)
        if 'row_signature' in df_existing.columns and row['row_signature'] in set(df_existing['row_signature'].astype(str)):
            print(f'Identical row already in {csv_path} — not appending.')
            return df_existing
        df_out = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_out = df_new
    df_out.to_csv(csv_path, index=False)
    print(f'Experiment log: {csv_path} ({len(df_out)} rows)')
    return df_out


def save_model_artifact():
    timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    sig = build_experiment_row()['row_signature'][:12]
    path = MODEL_OUTPUT_DIR / f'cnn_model_{timestamp}_{sig}.keras'
    model.save(path)
    print(f'Saved model to {path}')
    return path


model_path = save_model_artifact()
experiment_log_df = append_experiment_row()
experiment_log_df.tail(3)


Saved model to saved_models/cnn_model_2026-05-06_16-28-47_4c26ade6602a.keras
Experiment log: saved_models/experiment_results.csv (19 rows)


,timestamp,data_root,batch_test_root,sensor_cols_count,sensor_cols,resample_to_n_steps,apply_butterworth,butterworth_cutoff_hz,butterworth_order,sampling_rate_hz,...,final_val_loss,best_val_accuracy,best_val_loss,test_accuracy,test_loss,batch_test_accuracy,history_json,row_signature,data_paths,data_paths_count
16,2026-05-05 09:18:21,/home/jestin/ThesisRepo/ML/NewTrainingData/Dyn...,/home/jestin/ThesisRepo/ML/NewTestData/Joselyn...,176,"[""left_wrist_heading"",""left_wrist_pitch"",""left...",90,True,10.0,4,30.0,...,0.042075,0.988987,0.038306,1.000000,0.003532,1.000,"{""accuracy"":[0.7502756118774414,0.939360558986...",fbf22b437cc354983e9482e1e2e71946ec783760ffa234...,NaN,NaN
17,2026-05-06 16:26:14,NaN,/home/jestin/ThesisRepo/ML/NewTestData/Jestin/...,176,"[""left_wrist_heading"",""left_wrist_pitch"",""left...",90,True,10.0,4,30.0,...,0.226331,0.935897,0.135078,0.866667,0.377608,0.625,"{""accuracy"":[0.25641027092933655,0.54647433757...",e88a816fd825c4064d989addfd97b0c266ef022e6fd0cb...,"[""/home/jestin/ThesisRepo/ML/NewTestData/Alan/...",5.0
18,2026-05-06 16:28:47,NaN,/home/jestin/ThesisRepo/ML/NewTestData/Joselyn...,176,"[""left_wrist_heading"",""left_wrist_pitch"",""left...",90,True,10.0,4,30.0,...,0.063712,0.972678,0.063712,1.000000,0.032499,1.000,"{""accuracy"":[0.3786008358001709,0.592592597007...",4c26ade6602a78a76ab3a7fb9a38caaad42548e31ea87b...,"[""/home/jestin/ThesisRepo/ML/NewTestData/Alan/...",6.0


## 15. Export PDF report (parameters, paths, results)

Builds a compact PDF summarising the configuration, data locations, CV folds, hold-out and
batch-test results, and the saved model path. Uses only `reportlab` so it works in any env.


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
)
from reportlab.lib.enums import TA_LEFT


def _fmt(v, nd=4):
    if v is None:
        return '—'
    if isinstance(v, float):
        return f'{v:.{nd}f}'
    return str(v)


_VAL_STYLE = ParagraphStyle('KVValue', fontName='Helvetica', fontSize=7.5, leading=9.5,
                            textColor=colors.HexColor('#222222'))
_KEY_STYLE = ParagraphStyle('KVKey',   fontName='Helvetica-Bold', fontSize=7.5, leading=9.5,
                            textColor=colors.HexColor('#333333'))


def _wrap(text, style=_VAL_STYLE):
    if isinstance(text, Paragraph):
        return text
    s = str(text).replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
    return Paragraph(s, style)


def _kv_table(rows, col_widths=(50*mm, 135*mm)):
    wrapped = [[_wrap(r[0], _KEY_STYLE), _wrap(r[1], _VAL_STYLE)] for r in rows]
    tbl = Table(wrapped, colWidths=col_widths, hAlign='LEFT')
    tbl.setStyle(TableStyle([
        ('VALIGN',       (0, 0), (-1, -1), 'TOP'),
        ('BOTTOMPADDING',(0, 0), (-1, -1), 1),
        ('TOPPADDING',   (0, 0), (-1, -1), 1),
        ('LINEBELOW',    (0, 0), (-1, -2), 0.25, colors.HexColor('#dddddd')),
    ]))
    return tbl


def _section(title, styles):
    return Paragraph(f'<b>{title}</b>', styles['Section'])


def _build_comparison_chart(variant_results, out_path, batch_subject):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    labels    = [r['name'] for r in variant_results]
    cv_means  = [r['cv_mean'] if r['cv_mean'] is not None else 0 for r in variant_results]
    cv_stds   = [r['cv_std']  if r['cv_std']  is not None else 0 for r in variant_results]
    test_accs = [r['test_accuracy'] for r in variant_results]
    batch_acc = [r['batch_accuracy'] if r['batch_accuracy'] is not None else 0 for r in variant_results]
    x = np.arange(len(labels)); w = 0.27
    ax.bar(x - w, cv_means,  w, yerr=cv_stds, capsize=3, label=f'CV mean (k={CV_FOLDS})', color='#4c78a8')
    ax.bar(x,     test_accs, w, label='Hold-out', color='#f58518')
    ax.bar(x + w, batch_acc, w, label=f'Batch ({batch_subject})', color='#54a24b')
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=0)
    ax.set_ylim(0, 1.05); ax.set_ylabel('Accuracy')
    ax.set_title('1D CNN architecture variants — accuracy comparison')
    ax.legend(loc='lower right', fontsize=8); ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(test_accs):
        ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=7)
    plt.tight_layout()
    plt.savefig(out_path, dpi=140); plt.close()


def export_variants_report_pdf(out_path):
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name='Section',    parent=styles['Heading2'],
                              fontSize=11, spaceBefore=6, spaceAfter=2,
                              textColor=colors.HexColor('#1a3a6c')))
    styles.add(ParagraphStyle(name='SubSection', parent=styles['Heading3'],
                              fontSize=9.5, spaceBefore=4, spaceAfter=1,
                              textColor=colors.HexColor('#2a4a7c')))
    styles.add(ParagraphStyle(name='Small',      parent=styles['BodyText'],
                              fontSize=8, leading=10))
    styles['Title'].fontSize = 15
    styles['Title'].spaceAfter = 4
    styles['Title'].alignment = TA_LEFT

    doc = SimpleDocTemplate(
        str(out_path), pagesize=A4,
        leftMargin=12*mm, rightMargin=12*mm,
        topMargin=10*mm, bottomMargin=10*mm,
        title='1D CNN Variant Comparison',
    )
    story = []
    batch_subject = os.path.basename(os.path.dirname(str(BATCH_TEST_ROOT).rstrip('/'))) or 'batch'

    # Header
    story.append(Paragraph('1D CNN — Architecture Variant Comparison', styles['Title']))
    story.append(Paragraph(f"Generated {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles['Small']))
    story.append(Spacer(1, 4))

    # Data + config
    story.append(_section('Data &amp; configuration', styles))
    dp_rows = [[f'Training path {i+1}', p] for i, p in enumerate(ACTIVE_DATA_PATHS or [])]
    aug_enabled = ', '.join(n for n, c in AUGMENT_CONFIG.items() if c.get('enabled')) or 'no transforms enabled'
    dp_rows += [
        ['Batch-test root',     str(BATCH_TEST_ROOT)],
        ['Total trials',        f'{X.shape[0]} (train pool {X_train_pool.shape[0]}, hold-out {X_test.shape[0]})'],
        ['Classes',             f'{n_classes}: ' + ', '.join(le.classes_)],
        ['Sensor channels',     f'{n_channels} (sequence length {sequence_length})'],
        ['Resample / filter',   f'{RESAMPLE_TO_N_STEPS} steps · Butterworth cutoff {BUTTERWORTH_CUTOFF_HZ} Hz, order {BUTTERWORTH_ORDER}'],
        ['Normalisation',       str(NORMALISATION)],
        ['CV folds · shuffle',   f'{CV_FOLDS} · {CV_SHUFFLE}'],
        ['Augmentation',        ('on' if AUGMENT_TRAINING_DATA else 'off') + f' · copies={AUGMENTATION_COPIES_PER_SAMPLE} · ' + aug_enabled],
        ['Epochs · batch size',  f'{TRAIN_EPOCHS} · {TRAIN_BATCH_SIZE}'],
    ]
    story.append(_kv_table(dp_rows))

    # Comparison summary table
    story.append(_section('Variant comparison (summary)', styles))
    summary_rows = [['Variant', 'Params', f'CV mean (k={CV_FOLDS})', 'CV std',
                     'Hold-out acc', 'Hold-out loss', 'Batch acc', 'Batch n']]
    for r in variant_results:
        summary_rows.append([
            r['name'],
            f"{r['n_params']:,}",
            _fmt(r['cv_mean'], 4),
            _fmt(r['cv_std'], 4),
            _fmt(r['test_accuracy'], 4),
            _fmt(r['test_loss'], 4),
            _fmt(r['batch_accuracy'], 4) if r['batch_accuracy'] is not None else '—',
            f"{r['batch_correct']}/{r['batch_total']}" if r['batch_total'] else '—',
        ])
    best_test = max(variant_results, key=lambda r: r['test_accuracy'])['name']
    best_cv   = max(variant_results, key=lambda r: r['cv_mean'] or 0)['name']
    summary_t = Table(summary_rows, hAlign='LEFT',
                      colWidths=(22*mm, 22*mm, 24*mm, 18*mm, 24*mm, 22*mm, 22*mm, 18*mm))
    style_cmds = [
        ('FONT',         (0, 0), (-1, -1), 'Helvetica',     7.5),
        ('FONT',         (0, 0), (-1,  0), 'Helvetica-Bold', 7.5),
        ('BACKGROUND',   (0, 0), (-1,  0), colors.HexColor('#eef2f7')),
        ('GRID',         (0, 0), (-1, -1), 0.25, colors.HexColor('#cccccc')),
        ('VALIGN',       (0, 0), (-1, -1), 'MIDDLE'),
        ('ALIGN',        (1, 1), (-1, -1), 'CENTER'),
        ('BOTTOMPADDING',(0, 0), (-1, -1), 1),
        ('TOPPADDING',   (0, 0), (-1, -1), 1),
    ]
    for i, r in enumerate(variant_results, start=1):
        if r['name'] == best_test:
            style_cmds.append(('BACKGROUND', (0, i), (-1, i), colors.HexColor('#fff7e0')))
    summary_t.setStyle(TableStyle(style_cmds))
    story.append(summary_t)
    story.append(Spacer(1, 3))
    story.append(Paragraph(f'Best on hold-out: <b>{best_test}</b> · Best on CV: <b>{best_cv}</b>', styles['Small']))

    # Comparison chart
    cmp_chart = MODEL_OUTPUT_DIR / 'variants_comparison.png'
    _build_comparison_chart(variant_results, cmp_chart, batch_subject)
    img_w = 175*mm
    if cmp_chart.exists():
        story.append(Spacer(1, 4))
        story.append(Image(str(cmp_chart), width=img_w, height=img_w*0.5))

    # Training curves (already saved by curves cell)
    curves_path = MODEL_OUTPUT_DIR / 'training_curves.png'
    story.append(_section('Training curves (final retrain on full training pool)', styles))
    if curves_path.exists():
        story.append(Image(str(curves_path), width=img_w, height=img_w*0.36))

    # Per-variant detail pages
    for r in variant_results:
        story.append(PageBreak())
        story.append(Paragraph(f'Variant: {r["name"]}', styles['Title']))
        story.append(Paragraph(r['description'], styles['Small']))
        story.append(Spacer(1, 4))

        story.append(_kv_table([
            ['Trainable params',         f'{r["n_params"]:,}'],
            ['CV mean ± std',           f'{_fmt(r["cv_mean"])} ± {_fmt(r["cv_std"])}'],
            ['Hold-out test acc / loss', f'{_fmt(r["test_accuracy"])} · {_fmt(r["test_loss"])}'],
            ['Batch-test acc',
             f'{_fmt(r["batch_accuracy"])}  ({r["batch_correct"]}/{r["batch_total"]})'
             if r['batch_total'] else '—'],
        ]))

        if r['cv_fold_results']:
            story.append(Paragraph('<b>Cross-validation folds</b>', styles['SubSection']))
            ftbl = [['Fold', 'Train (post-aug)', 'Val', 'Val acc', 'Best val', 'Final val']]
            for fr in r['cv_fold_results']:
                ftbl.append([fr['fold'], fr['train_after_aug'], fr['val_samples'],
                             _fmt(fr['val_accuracy']),
                             _fmt(fr['best_val_accuracy']),
                             _fmt(fr['final_val_accuracy'])])
            t = Table(ftbl, hAlign='LEFT',
                      colWidths=(12*mm, 30*mm, 16*mm, 22*mm, 22*mm, 22*mm))
            t.setStyle(TableStyle([
                ('FONT',         (0, 0), (-1, -1), 'Helvetica',     7.5),
                ('FONT',         (0, 0), (-1,  0), 'Helvetica-Bold', 7.5),
                ('BACKGROUND',   (0, 0), (-1,  0), colors.HexColor('#eef2f7')),
                ('GRID',         (0, 0), (-1, -1), 0.25, colors.HexColor('#cccccc')),
                ('VALIGN',       (0, 0), (-1, -1), 'MIDDLE'),
                ('ALIGN',        (1, 1), (-1, -1), 'CENTER'),
                ('BOTTOMPADDING',(0, 0), (-1, -1), 1),
                ('TOPPADDING',   (0, 0), (-1, -1), 1),
            ]))
            story.append(t)

        # Per-class hold-out metrics
        story.append(Paragraph('<b>Per-class hold-out metrics</b>', styles['SubSection']))
        rep = r['classification_report']
        _cell = ParagraphStyle('Cell', fontName='Helvetica', fontSize=7, leading=8.5)
        cls_rows = [['Class', 'Precision', 'Recall', 'F1', 'Support']]
        for cls in le.classes_:
            d = rep.get(cls, {})
            cls_rows.append([_wrap(cls, _cell),
                             _fmt(d.get('precision'), 3),
                             _fmt(d.get('recall'), 3),
                             _fmt(d.get('f1-score'), 3),
                             int(d.get('support', 0))])
        for tag in ('macro avg', 'weighted avg'):
            d = rep.get(tag, {})
            cls_rows.append([tag,
                             _fmt(d.get('precision'), 3),
                             _fmt(d.get('recall'), 3),
                             _fmt(d.get('f1-score'), 3),
                             int(d.get('support', 0))])
        ct = Table(cls_rows, hAlign='LEFT', colWidths=(82*mm, 20*mm, 20*mm, 20*mm, 18*mm))
        ct.setStyle(TableStyle([
            ('FONT',         (0, 0), (-1, -1), 'Helvetica',     7),
            ('FONT',         (0, 0), (-1,  0), 'Helvetica-Bold', 7),
            ('FONT',         (0, -2), (-1, -1), 'Helvetica-Oblique', 7),
            ('BACKGROUND',   (0, 0), (-1,  0), colors.HexColor('#eef2f7')),
            ('GRID',         (0, 0), (-1, -1), 0.25, colors.HexColor('#cccccc')),
            ('VALIGN',       (0, 0), (-1, -1), 'MIDDLE'),
            ('ALIGN',        (1, 1), (-1, -1), 'CENTER'),
            ('BOTTOMPADDING',(0, 0), (-1, -1), 1),
            ('TOPPADDING',   (0, 0), (-1, -1), 1),
        ]))
        story.append(ct)

        # Batch-test per class
        if r.get('batch_df') is not None and len(r['batch_df']) > 0:
            story.append(Paragraph(f'<b>Batch test per class ({batch_subject})</b>', styles['SubSection']))
            bt_rows = [['Class', 'Correct', 'Total', 'Accuracy']]
            for cls, g in r['batch_df'].groupby('true'):
                c = int(g['correct'].sum()); n = int(len(g))
                bt_rows.append([_wrap(cls, _cell), c, n, _fmt(c/n if n else 0.0, 3)])
            bt_rows.append(['Overall', r['batch_correct'], r['batch_total'],
                            _fmt(r['batch_accuracy'], 3) if r['batch_accuracy'] is not None else '—'])
            bt = Table(bt_rows, hAlign='LEFT', colWidths=(102*mm, 20*mm, 20*mm, 22*mm))
            bt.setStyle(TableStyle([
                ('FONT',         (0, 0), (-1, -1), 'Helvetica',     7),
                ('FONT',         (0, 0), (-1,  0), 'Helvetica-Bold', 7),
                ('FONT',         (0, -1), (-1, -1), 'Helvetica-Bold', 7),
                ('BACKGROUND',   (0, 0), (-1,  0), colors.HexColor('#eef2f7')),
                ('BACKGROUND',   (0, -1), (-1, -1), colors.HexColor('#fff7e0')),
                ('GRID',         (0, 0), (-1, -1), 0.25, colors.HexColor('#cccccc')),
                ('VALIGN',       (0, 0), (-1, -1), 'MIDDLE'),
                ('ALIGN',        (1, 1), (-1, -1), 'CENTER'),
                ('BOTTOMPADDING',(0, 0), (-1, -1), 1),
                ('TOPPADDING',   (0, 0), (-1, -1), 1),
            ]))
            story.append(bt)

    doc.build(story)
    print(f'PDF report written to {out_path}')
    return out_path


# Unique filename so re-running never overwrites a previous report
_report_timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
_report_filename  = f'variants_report_{_report_timestamp}.pdf'
report_pdf_path   = export_variants_report_pdf(MODEL_OUTPUT_DIR / _report_filename)
